# 132STA - Direct stiffness method for a trusses

Apply the matrix approach to the direct stiffness method on a problem shown during the lecture.

### 0. Preparation

Run the setup and import cells below before starting the analysis. In Google Colab, the setup cell downloads the teaching functions; locally, it uses the repository on your computer.

In [ ]:
# Locate the teaching package (Colab or local Jupyter)
from pathlib import Path
import subprocess
import sys

repo = Path.cwd()
if not (repo / "sta132").is_dir():
    if (repo.parent / "sta132").is_dir():
        repo = repo.parent
    else:
        repo = Path("/content/132sta-python")
        if not (repo / "sta132").is_dir():
            subprocess.run(
                ["git", "clone", "--quiet", "--depth", "1",
                 "https://github.com/MartinDoskar/132sta-python.git", str(repo)],
                check=True,
            )
sys.path.insert(0, str(repo))
print("Using teaching functions from:", repo)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sta132.truss import stiffness_matrix_truss, axial_force_truss
from sta132.plotting import plot_truss
np.set_printoptions(precision=4, suppress=True)

### 1. Define the problem/structure

The structure is defined with coordinates of the joints and elements defined by their endpoints. 
Plus, define the material and cross-sectional characteristics.

In [ ]:
# Node coordinates [x, z] in metres
xz = np.array([[0., 0.],   # Node 1
               [0., 3.],   # Node 2
               [4., 3.]])  # Node 3

# Member endpoints, using zero-based Python node indices
bars = np.array([[0, 2],   # Member 1--3
                 [1, 2],   # Member 2--3
                 [1, 0]])  # Member 2--1

E = 200e6  # kN/m^2
A = 3e-3   # m^2


### 2. Member stiffness relations

Compute stiffness matrices of each truss element in the global coordinate system.

In [ ]:
K1, Kloc1, T1 = stiffness_matrix_truss(E, A, xz[0], xz[2])
K2, Kloc2, T2 = stiffness_matrix_truss(E, A, xz[1], xz[2])
K3, Kloc3, T3 = stiffness_matrix_truss(E, A, xz[1], xz[0])

print("K1:\n", K1)
print("K2:\n", K2)
print("K3:\n", K3)

### 3. Compatibility, equilibrium, and code numbers

Construct the global stiffness matrix using manually provided code numbers.

In [ ]:
K = np.zeros((6, 6))

code_numbers1 = np.array([1, 2, 5, 6]) - 1 # Member 1--3
code_numbers2 = np.array([3, 4, 5, 6]) - 1 # Member 2--3
code_numbers3 = np.array([3, 4, 1, 2]) - 1 # Member 2--1

# Convert the full code numbers to zero-based array indices.
K[np.ix_(code_numbers1, code_numbers1)] += K1
K[np.ix_(code_numbers2, code_numbers2)] += K2
K[np.ix_(code_numbers3, code_numbers3)] += K3

print("K [10^3 kN/m]:\n", K / 1e3)

### 4. Supports and applied load

Solve the system by extracting a subsystem pertinent to free DOFs.

In [ ]:
p = np.zeros(6)    # Applied nodal loads [kN]
p[4] = 270.        # Horizontal load at node 3 (code number 5)

d = np.zeros(6)    # Nodal displacements [m]

free_code_numbers = np.array([2, 5]) - 1
fixed_code_numbers = np.array([1, 3, 4, 6]) - 1

Kuu = K[np.ix_(free_code_numbers, free_code_numbers)]
print("Reduced K [10^3 kN/m]:\n", Kuu / 1e3)
print("Reduced load vector [kN]:", p[free_code_numbers])

d[free_code_numbers] = np.linalg.solve(Kuu, p[free_code_numbers])
print("All nodal displacements [mm]:", 1e3 * d)
print(f"w1 = {1e3*d[1]:.4f} mm, u3 = {1e3*d[4]:.4f} mm")

### 5. Support reactions

Obtain reactions by back-substitution into the global (unpartitioned) system.

In [ ]:
r = K @ d - p
print("Reaction vector [kN]:", r)
print("Support reactions [kN]:", r[fixed_code_numbers])
print("Residual at free DOFs [kN]:", r[free_code_numbers])

### 6. Member axial forces

Use solution to compute internal forces.

In [ ]:
N1 = axial_force_truss(E, A, xz[0], xz[2], d[code_numbers1])
N2 = axial_force_truss(E, A, xz[1], xz[2], d[code_numbers2])
N3 = axial_force_truss(E, A, xz[1], xz[0], d[code_numbers3])
N = np.array([N1, N2, N3])
print("Member forces [kN]:", N)

### 7. Plot the solution

Need scaling of the displacements

In [ ]:
fig, ax = plot_truss(xz, bars, d, scale=500)
plt.show()